# Mission - Construisez et testez une automatisation de transformation et analyse de données


## Contexte

Cette mission suit un scénario de projet professionnel. 


Vous pouvez suivre les étapes pour vous aider à réaliser vos livrables.

 

Avant de démarrer, nous vous conseillons de :

lire toute la mission et ses documents liés ;
prendre des notes sur ce que vous avez compris ;
consulter les étapes pour vous guider ; 
préparer une liste de questions pour votre première session de mentorat.
Prêt à mener la mission ?
Barres titres

 

Vous avez été récemment embauché comme Data Engineer dans l’entreprise BottleNeck, un marchand de vin prestigieux. 

 

Laurent, votre manager sur cette mission, vous accueille chaleureusement et vous propose de partager un café avec le reste de l’équipe. L’ambiance est bonne, et vous voilà déjà parfaitement intégré dans cette équipe détendue mais professionnelle. Vous rencontrez les responsables de produits : 

Laure, qui s'occupe des clients entreprises et des vins premium, 
Maria, qui s’occupe des particuliers et des vins ordinaires.
 

Après les présentations, effectuées dans une ambiance conviviale, Laurent vous explique le contexte data de l’entreprise : 

”Stéphane, notre data analyst a travaillé sur une première analyse de nos données. Il a présenté ses résultats lors de notre dernière réunion de COPIL et on a eu de bons retours de la part de nos responsables produits. 

Stéphane a travaillé sur ces 3 axes :  

Nettoyage des données provenant des 2 systèmes CMS et ERP.
Réconciliation de ces données afin de calculer le chiffre par produit et le chiffre d’affaires total réalisé.
Identification des vins premium avec utilisation des méthodes statistiques tel que le z-score et l’intervalle interquartile.
Ton rôle en tant que Data Engineer sera d’automatiser cette chaîne de traitement et d’analyse de données. 

Les responsables de données doivent recevoir les rapports des chiffres d’affaires tous les mois ainsi que les extractions des fichiers contenant les vins premium et les vins ordinaires. “

  

Fidèle à son habitude lors de l'arrivée d’un nouveau collaborateur, Laurent vous accompagne à votre poste de travail.

 

Il va vous faire suivre:

les données brutes par mail ; 
la méthode utilisée par Stéphane pour identifier les vins premium.
 

Vous recevez le mail de Laurent:

 

De : Laurent

À : moi

Objet : Exports tables et démarche de Stéphane

Re,

 

Tu trouveras ci-joints dans le fichier zip, les 2 exports du système ERP (erp.xlsx) et de la plateforme de vente en ligne CMS (web.xlsx) et le fichier liaison. Pour réconcilier les données, il faut passer par le fichier liaison (liaison.xlsx). 

 

Je te laisse prendre connaissance de ces éléments.

 

Voici la démarche de Stéphane : 

Suppression des valeurs manquantes et dédoublonnage des fichiers afin d’obtenir des clés primaires uniques avant les jointures entre les données.
Jointure des données communes entre ces deux fichiers propres en passant par le fichier intermédiaire liaison.xlsx.
Calcul des chiffres d’affaires par bouteille de vins et du chiffre d'affaires total sur les données fusionnées. 
Identification des vins premium en appliquant la méthode de z-score sur les prix des vins.
 

Ton rôle est d’industrialiser ce processus pour que les responsables produits puissent réaliser efficacement leurs ciblages marketing.

 

Au niveau de l’outil d'automatisation, notre DSI s’est positionné sur Kestra qui semble plus simple à prendre en main.

 

Nous souhaiterions présenter cette automatisation pour le prochain COPIL. Sur le plan technique, le DSI pourrait donner ses appréciations. Pourrais-tu préparer une présentation pour expliquer à l’ensemble de l’auditoire ta démarche et nous faire une démonstration ? L’objectif est de vulgariser la partie technique d’automatisation. 

 

Voici les éléments qui nous intéressent :

L’architecture de l’automatisation avec les procédures de tests. C'est-à-dire une conceptualisation des enchaînements des tâches de data transformation réalisées par Stéphane (nettoyages, jointures, agrégations, extractions) avec à l’issue de chaque tâche, une tâche de test pour vérifier que le résultat est juste.

L'implémentation de cette architecture avec les tests sur Kestra.

Une extraction du rapport au format Excel avec : 
* les chiffres d'affaires par produit ;
* le chiffre d'affaires total. 

Une extraction : 
* des vins premium ;
* des vins ordinaires.

Une planification de l'exécution de l’ensemble du workflow tous les 15 du mois à 9h.

Une solution si éventuellement il y avait des dysfonctionnements dans l’automatisation (indisponibilité d’un service tiers comme DuckDB par exemple).
 

N’hésite pas à solliciter Stéphane (sur la partie analyse de données) ou moi-même si tu as des questions.


Laurent

P.J. : 

Exports.zip
 

## Etape 1 - Concevez l'architecture d'automatisation (Data Lineage)



```mermaid

flowchart TD;
  DT[("Database")]
  e1["Import des données<br/>- erp.xlsx<br/>- liaison.xlsx<br/>- web.xlsx"]
  d1@{ shape: lean-r, label: "Données exportées" }
  A["Nettoyage:<br/>- suppression doublons<br/>- colonnes vides<br/>- lignes vides<br/>- formatage (date, texte, entiers...)"]
  d2@{ shape: lean-r, label: "Données nettoyées" }
  T1["Tests:<br/>- format de données,<br/>- unicité,<br/>- colonnes ou lignes vides"]
  c1{"Tests OK?"}
  V1["Vérifier les données"]
  B["Jointure:<br/>Jonction des tables erp et web<br/>via la table liaison"]
  d3@{ shape: lean-r, label: "Table Fusion" }
  T2["Tests:<br/>comparaison du nombre de lignes avant et après jonction"]
  c2{"Tests OK?"}
  V2["Vérifier les données"]
  C["Calcul du chiffre d'affaires (CA):<br/>- par vin,<br/>- total"]
  T3["Test de cohérence du CA:<br/>sommes des CA par vin vs CA total"]
  c3{"CA cohérent?"}
  V3["Vérifier les données"]
  D["Calcul du z-score sur le prix des vins"]
  E["Extraction rapport en Excel"]
  s1@{shape: doc, label: "Extrait CA par produit et total<br/>(fichier Excel)"}
  T4["Test de cohérence:<br/>prix vs seuil vin premium/vin ordinaire"]
  c4{"Test OK?"}
  V4["Vérifier les prix incohérents"]
  F["Séparation des données vins premium/ordinaires"]
  G-1["Extraction des données"]
  G-2["Extraction des données"]
  d4@{ shape: lean-r, label: "données vins premium" }
  d5@{ shape: lean-r, label: "données vins secondaires" }
  s2@{shape: doc, label: "Extrait vins premium<br/>(fichier CSV)"}
  s3@{shape: doc, label: "Extrait vins secondaires<br/>(fichier CSV)"}
  fin@{shape: terminal, label: "fin"}

  DT-->e1;
  e1-->d1;
  d1-->A;
  A-->d2;
  d2-->T1;
  T1-->c1;
  c1-->|"non"|V1;
  c1-->|"oui"| B;  
  V1-->A;  
  B-->d3;
  d3-->T2;
  T2-->c2;
  c2-->|"non"| V2;
  c2-->|"oui"| C;
  V2-->B;
  C-->T3;
  T3-->c3;
  c3-->|"non"| V3;
  c3-->|"oui"| D;
  c3--"oui"--> E;
  V3-->C;
  E-->s1;
  D-->T4;
  T4-->c4;
  c4-->|"non"| V4;
  c4-->|"oui"| F;
  V4-->D;
  F-->d4;
  F-->d5;
  d4-->G-1;
  d5-->G-2;
  G-1-->s2;
  G-2-->s3;
  s2-->fin;
  s3-->fin;
  
  style T1 fill:#fdebd0,stroke:#e67e22
  style T2 fill:#fdebd0,stroke:#e67e22
  style T3 fill:#fdebd0,stroke:#e67e22
  style T4 fill:#fdebd0,stroke:#e67e22
  style e1 fill:#d6eaf8,stroke:#2980b9
  style A fill:#d6eaf8,stroke:#2980b9  
  style B fill:#d6eaf8,stroke:#2980b9
  style C fill:#d6eaf8,stroke:#2980b9
  style D fill:#d6eaf8,stroke:#2980b9  
  style E fill:#d6eaf8,stroke:#2980b9
  style F fill:#d6eaf8,stroke:#2980b9
  style G-1 fill:#d6eaf8,stroke:#2980b9
  style G-2 fill:#d6eaf8,stroke:#2980b9
  style d1 fill:#d5f5e3,stroke:#27ae60
  style d2 fill:#d5f5e3,stroke:#27ae60
  style d3 fill:#d5f5e3,stroke:#27ae60
  style d4 fill:#d5f5e3,stroke:#27ae60
  style d5 fill:#d5f5e3,stroke:#27ae60
  style c1 fill:#ffdfe5,stroke:#ff5978
  style c2 fill:#ffdfe5,stroke:#ff5978
  style c3 fill:#ffdfe5,stroke:#ff5978
  style c4 fill:#ffdfe5,stroke:#ff5978
```

In [17]:
from IPython.display import IFrame

IFrame(src="Diagramme.drawio.html", width="100%", height="400")

Le pipeline démarre par l'import de trois fichiers sources depuis la base de données : erp.xlsx, liaison.xlsx et web.xlsx. Ces données brutes exportées passent d'abord par une étape de nettoyage, qui élimine les doublons, les colonnes et lignes vides, et harmonise le formatage (dates, chaînes de caractères, entiers). Un premier bloc de tests vérifie ensuite le format, l'unicité et l'absence de champs vides. Si un problème est détecté, les données repartent en arrière pour être corrigées ; sinon, le flux continue.

Vient ensuite la jointure des tables erp et web via la table de liaison, pour reconstituer une table fusionnée complète. Un deuxième contrôle compare le nombre de lignes avant et après la jonction, afin de s'assurer qu'aucune donnée n'a été perdue ou dupliquée par erreur.

Une fois les données validées, le pipeline calcule le chiffre d'affaires (par vin et au total), puis vérifie sa cohérence en comparant la somme des CA individuels au CA total. Ce chiffre d'affaires validé est alors exporté dans un rapport Excel.

En parallèle, un z-score est calculé sur les prix des vins pour distinguer statistiquement les vins premium des vins ordinaires. Un dernier test vérifie que cette classification respecte les seuils attendus. Les prix jugés incohérents sont réexaminés avant de poursuivre.

Enfin, les données sont séparées en deux groupes — vins premium et vins ordinaires — puis extraites séparément dans deux fichiers CSV distincts, marquant la fin du traitement.

## Etape 2 - Orchestrez les tâches nominales avec Kestra


### Installation de Kestra

Télécharger le docker-compose.yml nécessaire pour l'installation en tapant la commande suivante:
```
curl -o docker-compose.yml https://raw.githubusercontent.com/kestra-io/kestra/refs/heads/develop/docker-compose.yml
```
Le lancement de l'installation se fait via Docker avec la commande `docker compose up -d`

![installation_kestra.png](installation_kestra.png)

Une fois l'installation terminée, sur rendre dans un navigateur internet et taper l'URL http://localhost:8080/. La page ci-dessous apparaît pour la création du compte user admin:
![page_acceuil_kestra.png](page_acceuil_kestra.png)

Compléter le questionnaire:
![page_acceuil_kestra_questionnaire.png](page_acceuil_kestra_questionnaire.png)

Puis lancer l'interface UI Kestra:
![page_acceuil_kestra_start.png](page_acceuil_kestra_start.png)

L'interface se présente ainsi:
![page_acceuil_kestra_UI.png](page_acceuil_kestra_UI.png)

### Installation de DuckDB

Choix d'installation Duckdb via Python en utilisant la commande `pip install duckdb`

In [18]:
import duckdb

In [19]:
conn = duckdb.connect()

In [20]:
conn

In [21]:
conn.sql("SELECT 'world' as world;")

┌─────────┐
│  world  │
│ varchar │
├─────────┤
│ world   │
└─────────┘

Installation du CLI DuckDB en utilisant dans PowerShell la commande `winget install DuckDB.cli`

Lancement du CLI DuckDB:
```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10>duckdb
DuckDB v1.5.5 (Variegata)
Enter ".help" for usage hints.
memory D 
```


### Lecture des fichiers exports dans DuckDB

avec les commandes suivantes dans le CLI:
```
memory D FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_erp.xlsx' LIMIT 5;
┌────────────┬────────────┬────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │ price  │ stock_quantity │ stock_status │
│   double   │   double   │ double │     double     │   varchar    │
├────────────┼────────────┼────────┼────────────────┼──────────────┤
│     3847.0 │        1.0 │   24.2 │            0.0 │ outofstock   │
│     3849.0 │        1.0 │   34.3 │            0.0 │ outofstock   │
│     3850.0 │        1.0 │   20.8 │            0.0 │ outofstock   │
│     4032.0 │        1.0 │   14.1 │            0.0 │ outofstock   │
│     4039.0 │        1.0 │   46.0 │            0.0 │ outofstock   │
└────────────┴────────────┴────────┴────────────────┴──────────────┘
```

La lecture du fichier web.xlsx présente une erreur. 
```
memory D FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_web.xlsx' LIMIT 5;
Invalid Input Error:
read_xlsx: Failed to parse cell 'A198': Could not convert string 'bon-cadeau-25-euros' to DOUBLE
```

Utilisation de la commande suivante pour lire ce fichier:
```
memory D FROM read_xlsx('C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_web.xlsx', all_varchar=true) LIMIT 5;
┌─────────┬─────────┬──────────────┬──────────────┬────────────────┬─────────────┬────────────┬───────────┬───┬───────────────────────┬─────────────┬────────────────────────────┬────────────┬────────────┬────────────────┬───────────────┐
│   sku   │ virtual │ downloadable │ rating_count │ average_rating │ total_sales │ tax_status │ tax_class │ … │ post_content_filtered │ post_parent │            guid            │ menu_order │ post_type  │ post_mime_type │ comment_count │
│ varchar │ varchar │   varchar    │   varchar    │    varchar     │   varchar   │  varchar   │  varchar  │ … │        varchar        │   varchar   │          varchar           │  varchar   │  varchar   │    varchar     │    varchar    │
├─────────┼─────────┼──────────────┼──────────────┼────────────────┼─────────────┼────────────┼───────────┼───┼───────────────────────┼─────────────┼────────────────────────────┼────────────┼────────────┼────────────────┼───────────────┤
│ 16004   │ 0       │ 0            │ 0            │ 0              │ 5           │ NULL       │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ attachment │ image/jpeg     │ 0             │
│ NULL    │ 0       │ 0            │ 0            │ NULL           │ NULL        │ NULL       │ NULL      │ … │ NULL                  │ NULL        │ NULL                       │ NULL       │ NULL       │ NULL           │ NULL          │
│ 15075   │ 0       │ 0            │ 0            │ 0              │ 3           │ taxable    │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ product    │ NULL           │ 0             │
│ 16209   │ 0       │ 0            │ 0            │ 0              │ 6           │ taxable    │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ product    │ NULL           │ 0             │
│ 15763   │ 0       │ 0            │ 0            │ 0              │ 1           │ NULL       │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ attachment │ image/jpeg     │ 0             │
└─────────┴─────────┴──────────────┴──────────────┴────────────────┴─────────────┴────────────┴───────────┴───┴───────────────────────┴─────────────┴────────────────────────────┴────────────┴────────────┴────────────────┴───────────────┘
  5 rows                                                                                       use .last to show entire result                                                                                        28 columns (15 shown)
memory D
```
![DuckDB_lecturefichier_web.png](DuckDB_lecturefichier_web.png)


Lecture du fichier liaison:
```
memory D FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/fichier_liaison.xlsx', all_varchar=true)
         LIMIT 5;
┌────────────┬─────────┐
│ product_id │ id_web  │
│  varchar   │ varchar │
├────────────┼─────────┤
│ 3847       │ 15298   │
│ 3849       │ 15296   │
│ 3850       │ 15300   │
│ 4032       │ 19814   │
│ 4039       │ 19815   │
└────────────┴─────────┘
memory D
```

Création de la database dans DuckDB avec la commande suivante:
```
memory D ATTACH 'bottleneck.db';

```

Création et visualisation de la table `erp_export_data`
```
memory D CREATE TABLE bottleneck.erp_export_data AS (SELECT * FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_erp.xlsx');
memory D FROM bottleneck.erp_export_data LIMIT 5;
┌────────────┬────────────┬────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │ price  │ stock_quantity │ stock_status │
│   double   │   double   │ double │     double     │   varchar    │
├────────────┼────────────┼────────┼────────────────┼──────────────┤
│     3847.0 │        1.0 │   24.2 │            0.0 │ outofstock   │
│     3849.0 │        1.0 │   34.3 │            0.0 │ outofstock   │
│     3850.0 │        1.0 │   20.8 │            0.0 │ outofstock   │
│     4032.0 │        1.0 │   14.1 │            0.0 │ outofstock   │
│     4039.0 │        1.0 │   46.0 │            0.0 │ outofstock   │
└────────────┴────────────┴────────┴────────────────┴──────────────┘
```
![DuckDB_lecturefichier_creation_db.png](DuckDB_lecturefichier_creation_db.png)

Création des autres tables:

```
memory D CREATE TABLE bottleneck.web_export_data AS SELECT * FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/Fichier_web.xlsx', all_varchar=true);
memory D FROM botteneck.web_export_data LIMIT 5;
Catalog Error:
Table with name "botteneck.web_export_data" does not exist because schema "botteneck" does not exist.

LINE 1: FROM botteneck.web_export_data LIMIT 5;
             ^
memory D FROM bottleneck.web_export_data LIMIT 5;
```

```
memory D CREATE TABLE bottleneck.liaison_export_data AS SELECT * FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/fichier_liaison.xlsx', all_varchar=true);
memory D FROM bottleneck.liaison_export_data LIMIT 5;
┌────────────┬─────────┐
│ product_id │ id_web  │
│  varchar   │ varchar │
├────────────┼─────────┤
│ 3847       │ 15298   │
│ 3849       │ 15296   │
│ 3850       │ 15300   │
│ 4032       │ 19814   │
│ 4039       │ 19815   │
└────────────┴─────────┘
memory D
```
Vérifions la création de toutes les tables:
![showalltables.png](showalltables.png)



### Repérer les colonnes nulles ou complètement remplies de zéro

Avec la commande `SUMMARIZE` on peut identifier les colonnes nulles. 

**Pour l'export data Web:**
```
memory D CREATE VIEW bottleneck.view_web_export_data_summary AS SELECT * FROM (SUMMARIZE bottleneck.web_export_data);
```

La commande `SELECT * FROM bottleneck.view_web_export_data_summary;` permet de constater qu'il y a des colonnes qui sont entièrement nulles et d'autres qui ont uniquement des zéros.
![SUMMARIZE_web.png](SUMMARIZE_web.png)

```
memory D SELECT column_name FROM bottleneck.view_web_export_data_summary WHERE null_percentage = 100.00;
┌───────────────────────┐
│      column_name      │
│        varchar        │
├───────────────────────┤
│ tax_class             │
│ post_content          │
│ post_password         │
│ post_content_filtered │
└───────────────────────┘
memory D SELECT column_name FROM bottleneck.view_web_export_data_summary WHERE min='0' AND max='0';
┌────────────────┐
│  column_name   │
│    varchar     │
├────────────────┤
│ virtual        │
│ downloadable   │
│ rating_count   │
│ average_rating │
│ post_parent    │
│ menu_order     │
│ comment_count  │
└────────────────┘
memory D
```

Créer une nouvelle table `web_export_data_clean` qui ne contient pas ces colonnes:
```
memory D CREATE TABLE bottleneck.web_export_data_clean AS (
         SELECT * EXCLUDE ("virtual", downloadable, rating_count, average_rating, post_parent, menu_order, comment_count, tax_class, post_content, post_password, post_content_filtered) FROM bottleneck.web_export_data);

memory D DESCRIBE bottleneck.web_export_data_clean;
┌───────────────────────────┐
│   web_export_data_clean   │
│                           │
│ sku               varchar │
│ total_sales       varchar │
│ tax_status        varchar │
│ post_author       varchar │
│ post_date         varchar │
│ post_date_gmt     varchar │
│ post_title        varchar │
│ post_excerpt      varchar │
│ post_status       varchar │
│ comment_status    varchar │
│ ping_status       varchar │
│ post_name         varchar │
│ post_modified     varchar │
│ post_modified_gmt varchar │
│ guid              varchar │
│ post_type         varchar │
│ post_mime_type    varchar │
└───────────────────────────┘
memory D

```
Les colonnes exclues ne sont plus présentes.

**Pour l'export data erp:**

Le summarize indique qu'il n'y a pas de colonne 100% nulle ou remplie de zéros.
![SUMMARIZE_erp.png](SUMMARIZE_erp.png)

**Pour la table de liaison:**

Le summarize indique qu'il n'y a pas de colonne 100% nulle ou remplie de zéros.
![SUMMARIZE_liaison.png](SUMMARIZE_liaison.png)